### Tools
Models Can request to call tools that perform tasks such as fetching data from a database, searching the web or running code. Tools are fearing of : 
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON Schema) 
2. A function or coroutine to execute

In [6]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")

# response = model.invoke("why do I like girl?")
# response.content

In [7]:
from langchain.tools import tool
import requests

@tool
def getWeather(location: str) -> str:
    """Get current weather information for a city."""

    try:
        url = f"https://wttr.in/{location}?format=j1"

        response = requests.get(url, timeout=10)
        response.raise_for_status()

        data = response.json()

        current = data["current_condition"][0]
        area = data["nearest_area"][0]

        city = area["areaName"][0]["value"]
        country = area["country"][0]["value"]
        temperature = current["temp_C"]

    except requests.exceptions.RequestException:
        return f"Could not fetch weather information for {location}"
    except (KeyError, IndexError):
        return f"Could not find weather information for {location}"
    
    return (
        f"Weather in {city}, {country}\n"
        f"Temperature: {temperature}°C"
    )


model_with_tools = model.bind_tools([getWeather])

In [8]:
response = model_with_tools.invoke("What's the weather in Dubai?")
print(response)
for tool_call in response.tool_calls:
    # tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Tool: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks for weather in Dubai. Use getWeather function.', 'tool_calls': [{'id': 'fc_f6856439-237c-48a1-bb9f-7be075778a46', 'function': {'arguments': '{"location":"Dubai"}', 'name': 'getWeather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 127, 'total_tokens': 167, 'completion_time': 0.083790597, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.004881125, 'prompt_tokens_details': None, 'queue_time': 0.402363824, 'total_time': 0.088671722}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_90620edd96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0baab-671d-70c1-813e-9def87b7e1fa-0' tool_calls=[{'name': 'getWeather', 'args': {'location': 'Dubai'}, 'id': 'fc_f6856439-237c-48a1-bb9f-7be075778a46', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 12